In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
def simple_r2_score(y_true: np.ndarray, y_pred: np.ndarray):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return r2

In [ ]:
def weighted_r2_score(y_true: np.ndarray, y_pred: np.ndarray, weights: np.ndarray):
    ss_res = np.sum(weights * (y_true - y_pred) ** 2)
    y_bar = np.sum(weights * y_true) / np.sum(weights)
    ss_tot = np.sum(weights * (y_true - y_bar) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return r2


def single_target_weighted_r2_score(
    y_true: np.ndarray, y_pred: np.ndarray, weight: float
):
    weights = np.full_like(y_true, fill_value=weight)
    ss_res = np.sum(weights * (y_true - y_pred) ** 2)
    y_bar = np.sum(weights * y_true) / np.sum(weights)
    ss_tot = np.sum(weights * (y_true - y_bar) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return r2

In [ ]:
df = pd.read_csv("/kaggle/input/csiro-biomass/train.csv")
target_names = [
    "Dry_Green_g",
    "Dry_Dead_g",
    "Dry_Clover_g",
    "GDM_g",
    "Dry_Total_g",
]

y_true = np.empty((df.shape[0] // 5, len(target_names)))
for i, target_name in enumerate(target_names):
    y_true[:, i] = df[df["target_name"] == target_name]["target"].values


# y_trueのラベルに合わせて重みを設定
weights = np.array([0.1, 0.1, 0.1, 0.2, 0.5])
weights = np.tile(weights, (y_true.shape[0], 1))
print(weights.shape)
target_weight = 0.1

In [ ]:
# target_nameがDry_Dead_gのものを取り出す
plt.hist(y_true, bins=50)

In [ ]:
# min, macでランダムにy_predを生成
for i in range(y_true.shape[1]):
    y_pred_col = np.random.uniform(
        y_true[:, i].min(), y_true[:, i].max(), size=y_true[:, i].shape
    ).reshape(-1, 1)
    if i == 0:
        y_pred = y_pred_col
    else:
        y_pred = np.hstack((y_pred, y_pred_col))

    plt.figure(figsize=(8, 3))
    plt.subplot(1, 2, 1)
    plt.hist(y_pred_col, bins=50)
    plt.hist(y_true[:, i], bins=50, alpha=0.5)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.subplot(1, 2, 2)
    plt.scatter(y_true[:, i], y_pred_col, alpha=0.3)
    plt.scatter(y_true[:, i], y_true[:, i], color="red", marker=".", alpha=0.1)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.show()


# old_score = simple_r2_score(y_true, y_pred)
scores = []
for i in range(y_true.shape[1]):
    score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    scores.append(score * weights[0, i])
old_score = np.sum(scores) / np.sum(weights[0])
score = weighted_r2_score(y_true, y_pred, weights=weights)

print("Old R2 score:", old_score)
print("R2 score:", score)


# each target score
for i in range(y_true.shape[1]):
    each_old_score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    each_score = single_target_weighted_r2_score(
        y_true[:, i : i + 1], y_pred[:, i : i + 1], weight=weights[0, i]
    )
    print(f"Target index: {i}, target name: {target_names[i]}")
    print(f"\t Old R2 score\t: {each_old_score}")
    print(f"\t R2 score\t : {each_score}")

In [ ]:
# y_trueを少しノイズを加えたものをy_predとする
for i in range(y_true.shape[1]):
    y_pred_col = y_true[:, i : i + 1] + y_true[:, i : i + 1] * np.random.normal(
        0, 0.5, size=y_true[:, i : i + 1].shape
    )
    if i == 0:
        y_pred = y_pred_col
    else:
        y_pred = np.hstack((y_pred, y_pred_col))

    plt.figure(figsize=(8, 3))
    plt.subplot(1, 2, 1)
    plt.hist(y_pred_col, bins=50)
    plt.hist(y_true[:, i], bins=50, alpha=0.5)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.subplot(1, 2, 2)
    plt.scatter(y_true[:, i], y_pred_col, alpha=0.3)
    plt.scatter(y_true[:, i], y_true[:, i], color="red", marker=".", alpha=0.1)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.show()


# old_score = simple_r2_score(y_true, y_pred)
scores = []
for i in range(y_true.shape[1]):
    score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    scores.append(score * weights[0, i])
old_score = np.sum(scores) / np.sum(weights[0])
score = weighted_r2_score(y_true, y_pred, weights=weights)

print("Old R2 score:", old_score)
print("R2 score:", score)

# each target score
for i in range(y_true.shape[1]):
    each_old_score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    each_score = single_target_weighted_r2_score(
        y_true[:, i : i + 1], y_pred[:, i : i + 1], weight=weights[0, i]
    )
    print(f"Target index: {i}, target name: {target_names[i]}")
    print(f"\t Old R2 score\t: {each_old_score}")
    print(f"\t R2 score\t : {each_score}")


In [ ]:
# y_trueをランダムに小さくしたものをy_predとする
for i in range(y_true.shape[1]):
    y_pred_col = y_true[:, i : i + 1] * np.random.uniform(
        0.5, 1.0, size=y_true[:, i : i + 1].shape
    )
    if i == 0:
        y_pred = y_pred_col
    else:
        y_pred = np.hstack((y_pred, y_pred_col))

    plt.figure(figsize=(8, 3))
    plt.subplot(1, 2, 1)
    plt.hist(y_pred_col, bins=50)
    plt.hist(y_true[:, i], bins=50, alpha=0.5)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.subplot(1, 2, 2)
    plt.scatter(y_true[:, i], y_pred_col, alpha=0.3)
    plt.scatter(y_true[:, i], y_true[:, i], color="red", marker=".", alpha=0.1)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.show()


# old_score = simple_r2_score(y_true, y_pred)
scores = []
for i in range(y_true.shape[1]):
    score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    scores.append(score * weights[0, i])
old_score = np.sum(scores) / np.sum(weights[0])
score = weighted_r2_score(y_true, y_pred, weights=weights)

print("Old R2 score:", old_score)
print("R2 score:", score)

# each target score
for i in range(y_true.shape[1]):
    each_old_score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    each_score = single_target_weighted_r2_score(
        y_true[:, i : i + 1], y_pred[:, i : i + 1], weight=weights[0, i]
    )
    print(f"Target index: {i}, target name: {target_names[i]}")
    print(f"\t Old R2 score\t: {each_old_score}")
    print(f"\t R2 score\t : {each_score}")


In [ ]:
# y_predをy_trueのmeanに固定する
for i in range(y_true.shape[1]):
    y_pred_col = np.full_like(y_true[:, i : i + 1], fill_value=np.mean(y_true[:, i]))
    if i == 0:
        y_pred = y_pred_col
    else:
        y_pred = np.hstack((y_pred, y_pred_col))

    plt.figure(figsize=(8, 3))
    plt.subplot(1, 2, 1)
    plt.hist(y_pred_col, bins=50)
    plt.hist(y_true[:, i], bins=50, alpha=0.5)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.subplot(1, 2, 2)
    plt.scatter(y_true[:, i], y_pred_col, alpha=0.3)
    plt.scatter(y_true[:, i], y_true[:, i], color="red", marker=".", alpha=0.1)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.show()


# old_score = simple_r2_score(y_true, y_pred)
scores = []
for i in range(y_true.shape[1]):
    score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    scores.append(score * weights[0, i])
old_score = np.sum(scores) / np.sum(weights[0])
score = weighted_r2_score(y_true, y_pred, weights=weights)

print("Old R2 score:", old_score)
print("R2 score:", score)

# each target score
for i in range(y_true.shape[1]):
    each_old_score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    each_score = single_target_weighted_r2_score(
        y_true[:, i : i + 1], y_pred[:, i : i + 1], weight=weights[0, i]
    )
    print(f"Target index: {i}, target name: {target_names[i]}")
    print(f"\t Old R2 score\t: {each_old_score}")
    print(f"\t R2 score\t : {each_score}")


In [ ]:
# y_predをy_trueのmeanにランダムなノイズを加えたものにする
for i in range(y_true.shape[1]):
    y_pred_col = np.full_like(
        y_true[:, i : i + 1], fill_value=np.mean(y_true[:, i])
    ) + y_true[:, i : i + 1] * np.random.normal(0, 0.2, size=y_true[:, i : i + 1].shape)
    if i == 0:
        y_pred = y_pred_col
    else:
        y_pred = np.hstack((y_pred, y_pred_col))

    plt.figure(figsize=(8, 3))
    plt.subplot(1, 2, 1)
    plt.hist(y_pred_col, bins=50)
    plt.hist(y_true[:, i], bins=50, alpha=0.5)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.subplot(1, 2, 2)
    plt.scatter(y_true[:, i], y_pred_col, alpha=0.3)
    plt.scatter(y_true[:, i], y_true[:, i], color="red", marker=".", alpha=0.1)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.show()


# old_score = simple_r2_score(y_true, y_pred)
scores = []
for i in range(y_true.shape[1]):
    score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    scores.append(score * weights[0, i])
old_score = np.sum(scores) / np.sum(weights[0])
score = weighted_r2_score(y_true, y_pred, weights=weights)

print("Old R2 score:", old_score)
print("R2 score:", score)

# each target score
for i in range(y_true.shape[1]):
    each_old_score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    each_score = single_target_weighted_r2_score(
        y_true[:, i : i + 1], y_pred[:, i : i + 1], weight=weights[0, i]
    )
    print(f"Target index: {i}, target name: {target_names[i]}")
    print(f"\t Old R2 score\t: {each_old_score}")
    print(f"\t R2 score\t : {each_score}")

In [ ]:
# y_predをy_trueのmeedianに固定する
for i in range(y_true.shape[1]):
    y_pred_col = np.full_like(y_true[:, i : i + 1], fill_value=np.median(y_true[:, i]))
    if i == 0:
        y_pred = y_pred_col
    else:
        y_pred = np.hstack((y_pred, y_pred_col))

    plt.figure(figsize=(8, 3))
    plt.subplot(1, 2, 1)
    plt.hist(y_pred_col, bins=50)
    plt.hist(y_true[:, i], bins=50, alpha=0.5)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.subplot(1, 2, 2)
    plt.scatter(y_true[:, i], y_pred_col, alpha=0.3)
    plt.scatter(y_true[:, i], y_true[:, i], color="red", marker=".", alpha=0.1)
    plt.title(f"Target index: {i}, target name: {target_names[i]}")
    plt.show()


# old_score = simple_r2_score(y_true, y_pred)
scores = []
for i in range(y_true.shape[1]):
    score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    scores.append(score * weights[0, i])
old_score = np.sum(scores) / np.sum(weights[0])
score = weighted_r2_score(y_true, y_pred, weights=weights)

print("Old R2 score:", old_score)
print("R2 score:", score)

# each target score
for i in range(y_true.shape[1]):
    each_old_score = simple_r2_score(y_true[:, i : i + 1], y_pred[:, i : i + 1])
    each_score = single_target_weighted_r2_score(
        y_true[:, i : i + 1], y_pred[:, i : i + 1], weight=weights[0, i]
    )
    print(f"Target index: {i}, target name: {target_names[i]}")
    print(f"\t Old R2 score\t: {each_old_score}")
    print(f"\t R2 score\t : {each_score}")


In [ ]:
# y_predをy_trueの平均より10%小さくしたものをy_predとする
for rate in np.arange(0.1, 2.0, 0.1):
    for i in range(y_true.shape[1]):
        y_pred_col = y_true[:, i : i + 1] * rate
        if i == 0:
            y_pred = y_pred_col
        else:
            y_pred = np.hstack((y_pred, y_pred_col))

    score = weighted_r2_score(y_true, y_pred, weights=weights)
    print(f"R2 score (y_pred = y_true * {rate}):", score)


In [ ]:
# それぞれのmeanの値を確認
for i in range(y_true.shape[1]):
    print(
        f"Target index: {i}, target name: {target_names[i]}, mean: {np.mean(y_true[:, i])}"
    )

In [ ]:
# dfのtarget_nameがDry_Dead_gのものだけmeanの値とweighted_meanにする
dry_dead_mean = 12.04
dry_dead_predicts = df[df["target_name"] == "Dry_Dead_g"]["target"].values
df.loc[df["target_name"] == "Dry_Dead_g", "target"] = (
    dry_dead_mean * 0.9 + dry_dead_predicts * 0.1
)